In [1]:
!pip install --upgrade openai

In [2]:
!pip install --upgrade httpx brotli brotlicffi openai

## <함수 모음집>

In [3]:
import os
import requests
import xml.etree.ElementTree as ET
from dotenv import load_dotenv
from openai import OpenAI
from sklearn.linear_model import LinearRegression
import numpy as np
import json
import httpx
import time

load_dotenv()

# ── 설정 ──
KIPRIS_API_KEY = os.getenv("KIPRIS_API_KEY") 
KIPRIS_URL = "http://plus.kipris.or.kr/kipo-api/kipi/patUtiModInfoSearchSevice/getAdvancedSearch"

custom_http_client = httpx.Client(headers={"Accept-Encoding": "identity"})
openai_client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    http_client=custom_http_client,
)


# ── 함수 1: 슬롯 3개 → 특허검색용 키워드 변환 ──
def generate_patent_keyword(seed_interest: str, problem_to_solve: str, solution_approach: str) -> str:
    combined = f"{seed_interest} / {problem_to_solve} / {solution_approach}"
    response = openai_client.chat.completions.create(
        model="gpt-5-nano",
        messages=[{
            "role": "user",
            "content": f"""다음은 한 창업 아이디어의 핵심 요소입니다.
여기서 특허 검색에 쓸 핵심 기술 키워드를 딱 하나만, 최대한 짧게 뽑아줘.
반드시 2개의 한국어 단어 조합으로 답해 (예: "스마트팜 제어", "온도 센서"). 3개 이상 단어는 절대 쓰지 마.

다음 JSON 형식으로만 답해: {{"keyword": "여기에 키워드"}}

핵심 요소: {combined}""",
        }],
        max_completion_tokens=2000,
        reasoning_effort="low",
        response_format={"type": "json_object"},
    )
    result = json.loads(response.choices[0].message.content)
    return result["keyword"].strip()


# ── 함수 2: 키워드+연도 → 특허 출원건수 조회 ──
def get_patent_count(keyword: str, year: int, field: str = "astrtCont", max_retries: int = 3):
    params = {
        "ServiceKey": KIPRIS_API_KEY,
        field: keyword,
        "applicationDate": f"{year}0101~{year}1231",
        "patent": "true",
        "utility": "true",
        "numOfRows": "1",
        "pageNo": "1",
    }
    for attempt in range(max_retries):
        try:
            res = requests.get(KIPRIS_URL, params=params, timeout=10)
            root = ET.fromstring(res.content)

            success = root.find(".//header/successYN")
            if success is not None and success.text == "N":
                result_msg = root.find(".//header/resultMsg")
                msg = result_msg.text if result_msg is not None else "알 수 없는 오류"
                print(f"API 호출 실패 ({keyword}, {year}년): {msg}")
                return None

            total_count_el = root.find(".//count/totalCount")
            return int(total_count_el.text) if total_count_el is not None else 0

        except requests.exceptions.ConnectionError:
            if attempt < max_retries - 1:
                print(f"연결 끊김 ({keyword}, {year}년) — {attempt+1}번째 재시도 중...")
                time.sleep(2)
            else:
                print(f"연결 끊김 ({keyword}, {year}년) — {max_retries}번 재시도 후 포기")
                return None

# ── 함수 3: 과거 추이로 지정 연도(들)를 예측 (공개지연 구간 제외하고 학습) ──
def predict_future_years(year_count_dict, exclude_recent=2):
    """
    신뢰 가능한 마지막 연도 바로 다음 해(1년 뒤)만 예측
    """
    all_years = sorted(year_count_dict.keys())
    reliable_years = all_years[:-exclude_recent] if exclude_recent > 0 else all_years

    years = np.array(reliable_years).reshape(-1, 1)
    counts = np.array([year_count_dict[y] for y in reliable_years])

    if len(years) < 3:
        return {}

    model = LinearRegression().fit(years, counts)

    next_year = max(reliable_years) + 1   # 2024년까지가 신뢰가능이면 → 2025년 하나만
    predicted = max(0, model.predict([[next_year]])[0])

    return {next_year: round(float(predicted), 1)}


# ── 함수 4: 워크포워드 백테스트 — 이 모델이 과거에 얼마나 맞았는지 검증 ──
def backtest_forecast(year_count_dict: dict, min_train_years: int = 5) -> tuple:
    """
    과거 여러 시점을 돌아가며 '그 시점까지 데이터로 다음 해 예측 → 실제값과 비교'를 반복.
    반환: (시점별 결과 리스트, 평균절대오차 MAE)
    주의: 공개 지연 때문에 신뢰 못 하는 최근 연도는 호출 전에 이미 제외된 딕셔너리를 넣어야 함
    """
    years_sorted = sorted(year_count_dict.keys())
    results = []

    for i in range(min_train_years, len(years_sorted)):
        train_years = years_sorted[:i]
        test_year = years_sorted[i]

        X_train = np.array(train_years).reshape(-1, 1)
        y_train = np.array([year_count_dict[y] for y in train_years])

        model = LinearRegression()
        model.fit(X_train, y_train)

        predicted = max(0, model.predict([[test_year]])[0])
        actual = year_count_dict[test_year]
        error = abs(predicted - actual)

        results.append({
            "예측시점": test_year,
            "실제값": actual,
            "예측값": round(predicted, 1),
            "오차": round(error, 1),
        })

    if not results:
        return None, None

    mae = round(np.mean([r["오차"] for r in results]), 2)
    return results, mae


# ── 전체 파이프라인: 조회기간 확장 + 예측 + 백테스트까지 한 번에 ──
def get_patent_trend_with_forecast(seed_interest, problem_to_solve, solution_approach, past_years, exclude_recent=2):
    keyword = generate_patent_keyword(seed_interest, problem_to_solve, solution_approach)
    print(f"검색 키워드: {keyword}")

    actual = {}
    for yr in past_years:
        actual[yr] = get_patent_count(keyword, yr, field="astrtCont")
        time.sleep(0.3)

    reliable_years = sorted(actual.keys())[:-exclude_recent]
    reliable_actual = {y: actual[y] for y in reliable_years}

    forecast = predict_future_years(reliable_actual, exclude_recent=0)  # 이미 걸러진 걸 넣으니 0
    backtest_results, mae = backtest_forecast(reliable_actual, min_train_years=5)

    return {
        "keyword": keyword,
        "actual": actual,
        "reliable_years": reliable_years,
        "forecast": forecast,   # 이제 {2025: 값} 딱 하나만 나옴
        "backtest_results": backtest_results,
        "mae": mae,
    }

# # ── 테스트 ──
# if __name__ == "__main__":
#     result = get_patent_trend_with_forecast(
#         seed_interest="농업",
#         problem_to_solve="방울토마토 재배 시 온습도 관리가 어려움",
#         solution_approach="스마트팜 자동 제어 시스템",
#         past_years=list(range(2015, 2027)),   # 12개년으로 확장 (신뢰 가능 10개년)
#         n_future=1,
#         exclude_recent=2,
#     )

#     print("\n=== 결과 요약 ===")
#     print("키워드:", result["keyword"])
#     print("실측치:", result["actual"])
#     print("신뢰 가능 연도:", result["reliable_years"])
#     print("예측치:", result["forecast"])
#     print(f"평균절대오차(MAE): {result['mae']}")

#     print("\n=== 백테스트 상세 ===")
#     for r in result["backtest_results"]:
#         print(r)

## 키프리스 12대 특허 및 벤처기업 소규모 업종

In [4]:
strategic_to_keyword = {
    "반도체 디스플레이": "반도체 디스플레이",
    "이차전지": "이차전지",
    "첨단 모빌리티": "자율주행 시스템",
    "차세대 원자력": "핵융합",
    "첨단 바이오": "유전자 치료제",
    "우주항공 해양": "발사체",
    "수소": "수소",
    "사이버보안": "사이버보안",
    "인공지능": "인공지능",
    "차세대 통신": "이동통신",
    "첨단로봇 제조": "협동로봇",
    "양자": "양자컴퓨터",
}

industry_to_keyword = {
    "합성고무 제조업": "합성고무 조성물",
    "금속 스프링 제조업": "금속 스프링",
    "기타 석제품 제조업": "석재 가공",
    "인쇄잉크 및 회화용 물감 제조업": "인쇄잉크 조성물",
    "산업용 트럭 및 적재기 제조업": "지게차",
    "박판, 합판 및 유사 적층판 제조업": "적층판",
    "포장용 유리용기 제조업": "유리용기",
    "트레일러 및 세미트레일러 제조업": "트레일러",
    "내연기관 승용차 및 기타 여객용 자동차 제조업": "내연기관 자동차",
    "경성 인쇄회로기판 제조업": "인쇄회로기판",
    "목재 깔판류 및 기타 적재판 제조업": "파렛트",
    "전자코일, 변성기 및 기타 전자 유도자 제조업": "변압기 코일",
    "내연기관 제조업": "내연기관",
    "기타 기관 및 터빈 제조업": "터빈",
    "금속 주조 및 기타 야금용 기계 제조업": "금속 주조 장치",
    "신발 부분품 제조업": "신발 밑창",
    "윤활유 및 그리스 제조업": "윤활유 조성물",
    "전기 승용차 및 기타 여객용 전기 자동차 제조업": "전기자동차",
}

print(f"국가전략기술: {len(strategic_to_keyword)}개, 소규모: {len(industry_to_keyword)}개, 합계: {len(strategic_to_keyword)+len(industry_to_keyword)}개")

국가전략기술: 12개, 소규모: 18개, 합계: 30개


## 평균건수, MAE, 오차 계산

In [5]:
import pandas as pd

# 이미 API로 받아둔 원본 데이터를 재사용 — 같은 30개 카테고리 × 2015~2026년 데이터를
# 또 API로 재조회하면 KIPRIS 일일 호출 한도(LIMITED_NUMBER_OF_SERVICE_REQUESTS_EXCEEDS_ERROR)에
# 바로 걸리므로, MAE/오차 계산은 아래 원본 파일을 읽어서 수행한다.
raw_df = pd.read_csv(
    "../../../data/기술창업 분석 리포트/특허/특허_연도별_원본데이터_30개.xls",
    encoding="utf-8-sig",
)
year_cols = [c for c in raw_df.columns if c.isdigit()]

print(f"총 테스트 카테고리: {len(raw_df)}개 (원본 데이터 재사용 — API 미호출)\n")

final_results = []
for _, row in raw_df.iterrows():
    actual = {int(yr): int(row[yr]) for yr in year_cols}
    reliable_years = sorted(actual.keys())[:-2]
    reliable_actual = {y: actual[y] for y in reliable_years}
    backtest_results, mae = backtest_forecast(reliable_actual, min_train_years=5)

    if backtest_results is None:
        print(f"[{row['출처']}] {row['검색어']}: 백테스트 불가 (데이터 부족)")
        continue

    avg_count = np.mean(list(reliable_actual.values()))
    rel_error = round((mae / avg_count) * 100, 1) if avg_count > 0 else None

    final_results.append({
        "출처": row["출처"], "원래분류": row["원래분류"], "검색어": row["검색어"],
        "평균건수": round(avg_count, 1), "MAE": mae, "상대오차(%)": rel_error,
    })
    print(f"[{row['출처']}] {row['원래분류']} → '{row['검색어']}': 평균 {round(avg_count,1)}건, 오차 ±{rel_error}%")

총 테스트 카테고리: 30개 (원본 데이터 재사용 — API 미호출)

[국가전략기술] 반도체 디스플레이 → '반도체 디스플레이': 평균 136.5건, 오차 ±20.3%
[국가전략기술] 이차전지 → '이차전지': 평균 2120.1건, 오차 ±32.5%
[국가전략기술] 첨단 모빌리티 → '자율주행 시스템': 평균 278.4건, 오차 ±22.3%
[국가전략기술] 차세대 원자력 → '핵융합': 평균 14.9건, 오차 ±19.3%
[국가전략기술] 첨단 바이오 → '유전자 치료제': 평균 82.5건, 오차 ±12.3%
[국가전략기술] 우주항공 해양 → '발사체': 평균 58.9건, 오차 ±22.0%
[국가전략기술] 수소 → '수소': 평균 3133.1건, 오차 ±9.6%
[국가전략기술] 사이버보안 → '사이버보안': 평균 20.1건, 오차 ±22.5%
[국가전략기술] 인공지능 → '인공지능': 평균 1927.0건, 오차 ±15.3%
[국가전략기술] 차세대 통신 → '이동통신': 평균 1062.5건, 오차 ±9.5%
[국가전략기술] 첨단로봇 제조 → '협동로봇': 평균 25.2건, 오차 ±55.8%
[국가전략기술] 양자 → '양자컴퓨터': 평균 29.3건, 오차 ±55.8%
[소규모(벤처기업명단)] 합성고무 제조업 → '합성고무 조성물': 평균 20.5건, 오차 ±28.0%
[소규모(벤처기업명단)] 금속 스프링 제조업 → '금속 스프링': 평균 65.3건, 오차 ±12.6%
[소규모(벤처기업명단)] 기타 석제품 제조업 → '석재 가공': 평균 16.3건, 오차 ±37.5%
[소규모(벤처기업명단)] 인쇄잉크 및 회화용 물감 제조업 → '인쇄잉크 조성물': 평균 47.2건, 오차 ±21.6%
[소규모(벤처기업명단)] 산업용 트럭 및 적재기 제조업 → '지게차': 평균 97.5건, 오차 ±8.9%
[소규모(벤처기업명단)] 박판, 합판 및 유사 적층판 제조업 → '적층판': 평균 64.7건, 오차 ±13.2%
[소규모(벤처기업명단)] 포장용 유리용기 제조업 → '유리용기': 평

In [6]:
result_df = pd.DataFrame(final_results).dropna()

bins = [0, 25, 50, 100, 300, 100000]
labels = ["1~25건", "25~50건", "50~100건", "100~300건", "300건 이상"]
result_df["규모구간"] = pd.cut(result_df["평균건수"], bins=bins, labels=labels)

summary_table = result_df.groupby("규모구간")["상대오차(%)"].agg(["mean", "count"]).round(1)
summary_table.columns = ["평균 상대오차(%)", "표본 개수"]

print("=== 최종 기준선 (31개 실측 근거) ===")
print(summary_table)

print("\n=== 전체 상세 (규모순 정렬) ===")
print(result_df.sort_values("평균건수")[["출처","원래분류","검색어","평균건수","상대오차(%)"]].to_string())

result_df.to_csv("../../../data/기술창업 분석 리포트/특허/특허신뢰도_최종기준선.csv", index=False, encoding="utf-8-sig")
print("\n저장 완료")

=== 최종 기준선 (31개 실측 근거) ===


          평균 상대오차(%)  표본 개수
규모구간                       
1~25건           25.6      5
25~50건          32.6      7
50~100건         15.4      8
100~300건        18.7      3
300건 이상         13.5      7

=== 전체 상세 (규모순 정렬) ===
             출처                        원래분류        검색어    평균건수  상대오차(%)
3        국가전략기술                     차세대 원자력        핵융합    14.9     19.3
14  소규모(벤처기업명단)                  기타 석제품 제조업      석재 가공    16.3     37.5
26  소규모(벤처기업명단)       금속 주조 및 기타 야금용 기계 제조업   금속 주조 장치    19.2     20.7
7        국가전략기술                       사이버보안      사이버보안    20.1     22.5
12  소규모(벤처기업명단)                    합성고무 제조업   합성고무 조성물    20.5     28.0
10       국가전략기술                     첨단로봇 제조       협동로봇    25.2     55.8
11       국가전략기술                          양자      양자컴퓨터    29.3     55.8
20  소규모(벤처기업명단)   내연기관 승용차 및 기타 여객용 자동차 제조업   내연기관 자동차    30.5     14.8
28  소규모(벤처기업명단)               윤활유 및 그리스 제조업    윤활유 조성물    31.7     36.3
15  소규모(벤처기업명단)           인쇄잉크 및 회화용 물감 제조업   인쇄잉크 조성물    47

C:\Users\asia\AppData\Local\Temp\ipykernel_4708\801994661.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  summary_table = result_df.groupby("규모구간")["상대오차(%)"].agg(["mean", "count"]).round(1)


## 30개 카테고리 년도별 원본 데이터를 CSV로 저장 (기존 데이터 재사용, API 미호출)

In [7]:
# 위 MAE 계산에 쓴 raw_df를 그대로 CSV로 저장한다 (API 재호출 없음).
raw_df.to_csv(
    "../../../data/기술창업 분석 리포트/특허/특허_연도별_원본데이터_30개.csv",
    index=False, encoding="utf-8-sig",
)
print(f"저장 완료 — CSV 생성됨 ({len(raw_df)}행, {len(raw_df.columns)}열)")

저장 완료 — CSV 생성됨 (30행, 15열)


## ※ 참고: (특허청 12대 전략 + 벤처기업 소규모업종) 키프리스 사이트와 동일한지 검증
실제 키프리스 사이트와 검색 건수 동일하게 조회됨
단, 검색어 엔지니어링 필요하여 
'특허 시계열_특허 카테고리 표본 추출 (소규모 18개 업 - 벤처기업).ipynb'에 반영해놓음